In [2]:
import os
import sys

offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, sum, round, hour, avg, broadcast

spark = SparkSession.builder \
    .appName("SmartHome-DWH-Optimized-Monthly-Report") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

# 🔥 تنظيف الكاش تماماً لإجبار سبارك على رؤية الملفات الجديدة على القرص
spark.catalog.clearCache()

try:
    print("📂 جاري قراءة البيانات المعقمة من الباركيه الشامل...")
    historical_df = spark.read.parquet("/home/jovyan/work/storage/historical_parquet")
    
    db_url = "jdbc:postgresql://smarthome-postgres:5432/smarthome_energy"
    db_properties = {
        "user": "smarthome_user",
        "password": "smarthome_password",
        "driver": "org.postgresql.Driver"
    }
    
    print("📥 جاري جلب تسعيرة الكهرباء الحالية...")
    prices_df = spark.read.jdbc(url=db_url, table="electricity_prices", properties=db_properties)

    print("🧠 جاري استخراج مفاتيح الوقت...")
    historical_with_time = (historical_df
        .withColumn("hour_of_day", hour(col("timestamp")))
        .withColumn("date_hour_key", substring(col("timestamp").cast("string"), 1, 13))
        .withColumn("report_month", substring(col("timestamp").cast("string"), 1, 7))
    )

    # 👀 سطر فحص: لنرى ما هي الشهور الموجودة فعلياً في ملفات الباركيه الآن!
    print("🧐 الشهور المكتشفة داخل ملف الباركيه حالياً هي:")
    historical_with_time.select("report_month").distinct().show()

    print("🔄 [المرحلة 1] تجميع متوسط الاستهلاك لكل جهاز في الساعة (لمنع التكرار)...")
    hourly_device_avg = (historical_with_time
        .groupBy("report_month", "date_hour_key", "hour_of_day", "device_id")
        .agg(avg("power_consumption_watts").alias("avg_watts_per_hour"))
    )

    enriched_df = hourly_device_avg.join(broadcast(prices_df), hourly_device_avg.hour_of_day == prices_df.hour, "left")

    print("💰 [المرحلة 2] حساب الـ kWh والتكلفة لكل سطر ساعي...")
    calculated_df = (enriched_df
        .withColumn("real_kwh", col("avg_watts_per_hour") / 1000.0)
        .withColumn("real_cost", col("real_kwh") * col("price_per_kwh"))
    )

    print("📈 [المرحلة 3] التجميع الشهري الشامل الحقيقي...")
    monthly_report_df = (calculated_df
        .groupBy("report_month")
        .agg(
            round(sum("real_kwh"), 2).cast("decimal(12,2)").alias("total_kwh"),
            round(sum("real_cost"), 2).cast("decimal(15,2)").alias("estimated_cost_yer")
        )
        .orderBy("report_month") # ترتيب الشهور تصاعدياً
        .select("report_month", "total_kwh", "estimated_cost_yer")
    )

    print("📥 جاري حقن التقارير الشهرية المتطابقة في قاعدة البيانات...")
    monthly_report_df.write \
        .mode("overwrite") \
        .option("truncate", "true") \
        .jdbc(url=db_url, table="dwh_monthly_energy_reports", properties=db_properties)
        
    print("✅ تم التحديث بنجاح وتطابقت الأسعار تماماً!")
    monthly_report_df.show(10, truncate=False) # رفعنا العرض لـ 10 أسطر لرؤية كل الشهور

except Exception as e:
    print(f"❌ فشل تحديث البيانات في التقرير الشهري: {e}")
finally:
    spark.stop()

📂 جاري قراءة البيانات المعقمة من الباركيه الشامل...
📥 جاري جلب تسعيرة الكهرباء الحالية...
🧠 جاري استخراج مفاتيح الوقت...
🧐 الشهور المكتشفة داخل ملف الباركيه حالياً هي:
+------------+
|report_month|
+------------+
|     2026-07|
+------------+

🔄 [المرحلة 1] تجميع متوسط الاستهلاك لكل جهاز في الساعة (لمنع التكرار)...
💰 [المرحلة 2] حساب الـ kWh والتكلفة لكل سطر ساعي...
📈 [المرحلة 3] التجميع الشهري الشامل الحقيقي...
📥 جاري حقن التقارير الشهرية المتطابقة في قاعدة البيانات...
✅ تم التحديث بنجاح وتطابقت الأسعار تماماً!
+------------+---------+------------------+
|report_month|total_kwh|estimated_cost_yer|
+------------+---------+------------------+
|2026-07     |7.50     |2186.87           |
+------------+---------+------------------+



In [3]:
import os
import sys

# 1. إعداد مسارات المكتبات لتعمل أوفلاين
offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, sum, round, hour, when, to_timestamp, broadcast, avg, first
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

# 2. بناء الجلسة
spark = SparkSession.builder \
    .appName("SmartHome-DWH-Parquet-Incremental-Analytics-Safe") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

# 💡 تعريف المخطط (Schema)
parquet_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("house_type", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("is_room_occupied", BooleanType(), True), 
    StructField("power_consumption_watts", DoubleType(), True),
    StructField("status", StringType(), True)
])

try:
    print("📂 [بث تزايدي] جاري مراقبة وقراءة البيانات الجديدة فقط من الباركيه...")
    historical_stream_df = spark.readStream \
        .schema(parquet_schema) \
        .parquet("/home/jovyan/work/storage/historical_parquet")
    
    db_url = "jdbc:postgresql://smarthome-postgres:5432/smarthome_energy"
    db_properties = {
        "user": "smarthome_user",
        "password": "smarthome_password",
        "driver": "org.postgresql.Driver"
    }

    print("📥 جاري جلب تسعيرة الكهرباء لربطها ديناميكياً...")
    prices_df = spark.read.jdbc(url=db_url, table="electricity_prices", properties=db_properties)

    print("🧠 جاري هندسة وتجهيز مفاتيح الوقت الفعلي الأصلي...")
    
    prepared_stream_df = (historical_stream_df
        .withColumn("hour_of_day", hour(col("timestamp")))
        .withColumn("date_hour_key", substring(col("timestamp").cast("string"), 1, 13))
        .withColumn("report_date", to_timestamp(substring(col("timestamp").cast("string"), 1, 10), "yyyy-MM-dd"))
        .withColumn("is_peak_hours", when((hour(col("timestamp")) >= 18) & (hour(col("timestamp")) <= 23), True).otherwise(False))
    )

    # دالة المعالجة اليومية الذكية والمصححة رياضياً وهندسياً
    def write_to_postgres(batch_df, batch_id):
        if batch_df.count() == 0:
            return
            
        print(f"📊 [الدفعة {batch_id}] جاري معالجة الحسابات اليومية بدقة رياضية متطابقة...")
        
        # 1. التجميع الأساسي على مستوى الساعة والجهاز فقط (منع التضخيم المالي نهائياً)
        # نستخدم first() لجلب الخصائص الوصفية للمنطقة ونوع الجهاز دون تفتيت السطور
        hourly_core = (batch_df
            .groupBy("report_date", "date_hour_key", "hour_of_day", "is_peak_hours", "device_id")
            .agg(
                avg("power_consumption_watts").alias("avg_watts"),
                first("zone").alias("zone"),
                first("device_type").alias("device_type"),
                first("is_room_occupied").alias("is_room_occupied"),
                first("status").alias("status")
            )
        )
        
        # 2. ربط الأسعار الصافي
        joined_batch = hourly_core.join(broadcast(prices_df), hourly_core.hour_of_day == prices_df.hour, "left")
        
        # 3. الحسابات الصافية الصحيحة للساعة
        calculated_batch = (joined_batch
            .withColumn("real_kwh", col("avg_watts") / 1000.0) 
            .withColumn("real_cost", col("real_kwh") * col("price_per_kwh"))
        )
        
        # 4. الحسابات السلوكية والأعطال المبنية على الخصائص المستخرجة
        analyzed_batch = (calculated_batch
            .withColumn("wasted_kwh", when((col("is_room_occupied") == False) & (col("avg_watts") > 0.0), col("real_kwh")).otherwise(0.0))
            .withColumn("is_sensor_fault", when(col("status") == "SENSOR_FAULT", 1).otherwise(0))
            .withColumn("is_lost_signal", when(col("status") == "LOST_SIGNAL", 1).otherwise(0))
        )
        
        # 5. التجميع اليومي النهائي للمستودع
        final_analytics_df = (analyzed_batch.groupBy("report_date", "zone", "device_type", "is_room_occupied", "is_peak_hours")
            .agg(
                sum("real_kwh").cast("double").alias("raw_kwh_sum"),
                sum("real_kwh").cast("double").alias("total_power_kwh"),
                round(sum("real_cost"), 2).cast("double").alias("estimated_cost_yer"),
                round(sum("wasted_kwh"), 2).cast("double").alias("wasted_energy_kwh"),
                sum("is_sensor_fault").cast("long").alias("sensor_fault_count"),
                sum("is_lost_signal").cast("long").alias("lost_signal_count")
            )
        ).select(
            "report_date", "zone", "device_type", "is_room_occupied", "is_peak_hours",
            "raw_kwh_sum", "total_power_kwh", "estimated_cost_yer", "wasted_energy_kwh",
            "sensor_fault_count", "lost_signal_count"
        )
            
        final_analytics_df.write \
            .mode("append") \
            .jdbc(url=db_url, table="dwh_energy_analytics", properties=db_properties)

    # مسار حفظ الإحداثيات للتشغيل التزايدي
    checkpoint_path = "/home/jovyan/work/storage/checkp_analytics_increment"
    
    print("📥 جاري تشغيل المحرك التزايدي اليومي الصائب...")
    query = prepared_stream_df.writeStream \
        .foreachBatch(write_to_postgres) \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(availableNow=True) \
        .start()
        
    query.awaitTermination()
    print("✅ [نجاح باهر] تم الحقن اليومي الموزون وتطابق الحسابات بنجاح!")

except Exception as e:
    print(f"❌ فشل في محرك تحليلات الباركيه اليومي: {e}")

finally:
    spark.stop()

📂 [بث تزايدي] جاري مراقبة وقراءة البيانات الجديدة فقط من الباركيه...
📥 جاري جلب تسعيرة الكهرباء لربطها ديناميكياً...
🧠 جاري هندسة وتجهيز مفاتيح الوقت الفعلي الأصلي...
📥 جاري تشغيل المحرك التزايدي اليومي الصائب...
✅ [نجاح باهر] تم الحقن اليومي الموزون وتطابق الحسابات بنجاح!


import shutil

checkpoint_path = "/home/jovyan/work/storage/checkp_analytics_increment"

try:
    # مسح المجلد بجميع محتوياته بأمان
    shutil.rmtree(checkpoint_path, ignore_errors=True)
    print("🗑️ تم مسح مجلد الـ Checkpoint وتصفيره بنجاح ساحق!")
except Exception as e:
    print(f"❌ فشل مسح المجلد: {e}")

import os
import sys
from datetime import datetime, timedelta
import random

# إعداد مسارات المكتبات لتعمل أوفلاين
offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
# تم إزالة round من هنا لمنع التضارب مع دالة بايثون الأصلية
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

# 1. بناء الجلسة
spark = SparkSession.builder \
    .appName("SmartHome-Data-Generator-Past-3-Months") \
    .getOrCreate()

print("🕒 جاري توليد بيانات الأشهر الثلاثة السابقة (أبريل، مايو، يونيو 2026)...")

# إعداد الـ Schema المتطابق
parquet_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("house_type", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("is_room_occupied", BooleanType(), True), 
    StructField("power_consumption_watts", DoubleType(), True),
    StructField("status", StringType(), True)
])

# الأجهزة والمناطق للمحاكاة
devices = [
    {"id": "LV_AC", "type": "AC", "zone": "Living_Room", "watts": 1100.0},
    {"id": "MB_AC", "type": "AC", "zone": "Master_Bedroom", "watts": 850.0},
    {"id": "KT_Heavy", "type": "Heavy_Appliance", "zone": "Kitchen", "watts": 1300.0},
    {"id": "LV_Light", "type": "Lighting", "zone": "Living_Room", "watts": 15.0}
]

raw_data = []

# تحديد نقطة البداية: 1 أبريل 2026
start_past_date = datetime(2026, 4, 1, 0, 0, 0)
# عدد الساعات الإجمالية لـ 3 أشهر (أبريل 30 يوم + مايو 31 يوم + يونيو 30 يوم = 91 يوماً)
total_hours = 24 * 91 

print(f"⏳ جاري جدولة الحسابات لـ {total_hours} ساعة...")

for hour_offset in range(0, total_hours):
    current_time = start_past_date + timedelta(hours=hour_offset)
    timestamp_str = current_time.strftime("%Y-%m-%d %H:%M:%S")
    
    for dev in devices:
        is_occupied = random.choice([True, False])
        # محاكاة استهلاك طاقة واقعي بناءً على حالة الغرفة واستخدام الدالة الأصلية بشكل آمن
        power = round(random.uniform(dev["watts"] * 0.6, dev["watts"] * 1.1), 2) if is_occupied else 0.0
        status = "ON" if power > 0 else "OFF"
        
        raw_data.append((
            timestamp_str,
            "Smart_Luxury_Villa",
            "YER",
            dev["zone"],
            dev["id"],
            dev["type"],
            is_occupied,
            float(power),
            status
        ))

# 2. تحويل البيانات إلى DataFrame وحفظها
print("💾 جاري تحويل البيانات وضخها في مجلد الباركيه...")
generated_df = spark.createDataFrame(raw_data, schema=parquet_schema)

output_parquet_path = "/home/jovyan/work/storage/historical_parquet"

# حفظ بنظام append لتنضم إلى جانب الملفات الحالية
generated_df.write \
    .mode("append") \
    .parquet(output_parquet_path)

print(f"✅ تم بنجاح حقن الأشهر الثلاثة السابقة في الباركيه! إجمالي السطور: {generated_df.count()}")
spark.stop()